# AVF Acoustic Analysis: Machine Learning Classification (ver.2.3)

**変更点（v4から）:**
- `MODELS_TO_RUN`：`['RF', 'LR']`、`['RF']`、`['LR']` のいずれかを選択
- `RUN_MODE`：`'full'`、`'cv_only'`、`'test_only'` のいずれかを選択
- ハイパーパラメータ探索グリッドは Cell 2 で設定可能

**仕様:**
- ネスト付き CV：外部フォールドに `StratifiedGroupKFold` を使用
- 各外部フォールド内で特徴選択を1回実行し、その後 `MODELS_TO_RUN` で指定したモデルを評価
- RF の HP は OOB-AUC で選択
- LR の HP は `GridSearchCV` + `Pipeline` + `GroupKFold` で AUC を評価
- テスト評価は、保持したテストセットで HP チューニング済みモデルを実行

In [ ]:
import sys
import pandas as pd
import numpy as np
import re
import warnings
import time
from pathlib import Path
from collections import Counter
from itertools import product as iter_product

if "google.colab" in sys.modules:
    !pip install python-docx statsmodels shap openpyxl -q
    !apt-get -qq install -y fonts-liberation
    from google.colab import drive
    drive.mount('/content/drive')

from sklearn.model_selection import (
    train_test_split, GroupKFold, StratifiedGroupKFold, GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, roc_auc_score, RocCurveDisplay,
    brier_score_loss
)
from sklearn.feature_selection import RFE
from sklearn.base import clone
from sklearn.calibration import CalibrationDisplay
from sklearn.compose import make_column_selector, ColumnTransformer

from scipy.stats import mannwhitneyu, chi2_contingency, norm
import matplotlib.pyplot as plt
from docx import Document
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Liberation Sans'

In [ ]:
# ======================================
# Configuration
# ======================================
TARGET_COLUMN_FOR_LABELING = 'FV'  # 'FV' or 'RI'
GROUP_THRESHOLD = 400              # FV: 400, RI: 0.6

DRIVE_BASE_PATH = Path("/path/to/your/data")
INPUT_EXCEL_PATH = DRIVE_BASE_PATH / "your_data.xlsx"
OUTPUT_DIR = DRIVE_BASE_PATH / f"{TARGET_COLUMN_FOR_LABELING}_{GROUP_THRESHOLD}_results_phase1"

# Run Options #####
MODELS_TO_RUN = ['RF', 'LR']  # ['RF', 'LR'], ['RF'], or ['LR']
RUN_MODE = 'full'             # 'full', "cv_only" or 'test_only'

# Analysis Parameters #####
N_FEATURES_TO_SELECT = 5
N_SPLITS_CV = 5
N_BOOTSTRAPS_STABILITY = 500
N_BOOTSTRAPS_CI = 1000
FORCE_RESELECT = False  # True: always re-run, False: load from cache if exists
UNIFIED_RANDOM_STATE = 243

# RF Hyperparameter Search Space ####
RF_PARAM_GRID = {
    'n_estimators': [100, 300, 500, 1000],
    'max_depth':[None, 3, 5],  # [None, 3, 5],
    'min_samples_leaf': [1, 3]  # [1, 3]
}

# LR Hyperparameter Search Space ####
LR_PARAM_GRID = {
    'model__C': [0.01, 0.1, 1.0, 10]
}

print(f"Task: {TARGET_COLUMN_FOR_LABELING} >= {GROUP_THRESHOLD}")
print(f"Models: {MODELS_TO_RUN}")
print(f"Run mode: {RUN_MODE}")
if 'RF' in MODELS_TO_RUN:
    print(f"RF HP grid: {len(list(iter_product(*RF_PARAM_GRID.values())))} combinations")
if 'LR' in MODELS_TO_RUN:
    print(f"LR HP grid: {len(LR_PARAM_GRID['model__C'])} values of C")

In [ ]:
# ====================================
# Data Loading & Preprocessing
# ====================================
def load_and_preprocess_data(file_path: Path, target_column: str, threshold: float):
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    df = pd.read_excel(file_path)
    target_variable = f'{target_column}>={threshold}'
    df[target_variable] = (df[target_column] >= threshold).astype(int)

    print(f"Loaded data from '{file_path.name}'")
    print(f"Created binary target '{target_variable}' from '{target_column}'")
    print(f"n={len(df)} samples")

    return target_variable, df


def split_data_by_patient(df: pd.DataFrame, target_variable: str,
                          test_size: float = 0.2, random_state: int = None):
    unique_patients = df['Pt No'].unique()
    patient_labels = df.drop_duplicates(subset=['Pt No']).set_index('Pt No')[target_variable]

    train_patients, test_patients = train_test_split(
        unique_patients, test_size=test_size, random_state=random_state,
        stratify=patient_labels.reindex(unique_patients)
    )

    train_indices = df[df['Pt No'].isin(train_patients)].index
    test_indices = df[df['Pt No'].isin(test_patients)].index

    print(f"\nPatient-level split: Train={len(train_patients)}, Test={len(test_patients)}")
    print(f"Sample-level split: Train={len(train_indices)}, Test={len(test_indices)}")

    for name, indices in [("Training", train_indices), ("Test", test_indices)]:
        dist = df.loc[indices, target_variable].value_counts(normalize=True).sort_index()
        print(f"\n{name} data ({len(indices)} samples):")
        for label, prop in dist.items():
            count = df.loc[indices, target_variable].value_counts().get(label, 0)
            print(f"  Class {label}: {count} ({prop:.2%})")

    return train_indices, test_indices

In [ ]:
# ==================================
# Table Creation Functions
# ==================================
def create_patient_characteristics_table(df: pd.DataFrame, target_variable: str) -> pd.DataFrame:
    print("\n--- Creating Table 1: Patient Characteristics ---")
    df_patients = df.drop_duplicates(subset=['Pt No']).copy()

    group0 = df_patients[df_patients[target_variable] == 0]
    group1 = df_patients[df_patients[target_variable] == 1]

    table_data = []

    continuous_vars = ['age', 'POD', 'FV', 'RI', 'RA diameter', 'shunt diameter']
    for var in continuous_vars:
        if var not in df.columns:
            continue

        n_missing0 = group0[var].isnull().sum()
        n_missing1 = group1[var].isnull().sum()

        mean0, std0 = group0[var].mean(), group0[var].std()
        mean1, std1 = group1[var].mean(), group1[var].std()

        mean0_str = f"{mean0:.1f} ± {std0:.1f}"
        if n_missing0 > 0:
            mean0_str += f" (missing: {n_missing0})"

        mean1_str = f"{mean1:.1f} ± {std1:.1f}"
        if n_missing1 > 0:
            mean1_str += f" (missing: {n_missing1})"

        _, p_val = mannwhitneyu(group0[var].dropna(), group1[var].dropna())

        table_data.append({
            'Characteristic': var,
            f'{TARGET_COLUMN_FOR_LABELING} < {GROUP_THRESHOLD} (n={len(group0)})': mean0_str,
            f'{TARGET_COLUMN_FOR_LABELING} >= {GROUP_THRESHOLD} (n={len(group1)})': mean1_str,
            'P-Value': f"{p_val:.2f}" if p_val >= 0.01 else "<0.01"
        })

    categorical_vars = {
        'men': 'Male Sex (n, %)',
        'arterial calcification': 'Arterial Calcification (n, %)',
        'DM': 'DM (n, %)',
        'HTN': 'HTN (n, %)',
        'Heart disease': 'Heart Disease (n, %)',
        'tabaco site': 'tabaco site (n, %)',
        'left': 'left (n, %)'
    }

    mapping_dict = {'y': 1, 'n': 0, 'male': 1, 'female': 0, '1': 1, '0': 0, '1.0': 1, '0.0': 0}

    for var, name in categorical_vars.items():
        if var not in df_patients.columns:
            continue

        df_patients[var + '_numeric'] = df_patients[var].astype(str).str.lower().map(mapping_dict)
        df_filtered = df_patients.dropna(subset=[var + '_numeric'])

        group0_cat = df_filtered[df_filtered[target_variable] == 0]
        group1_cat = df_filtered[df_filtered[target_variable] == 1]

        crosstab = pd.crosstab(df_filtered[var + '_numeric'], df_filtered[target_variable])

        p_val = chi2_contingency(crosstab)[1] if crosstab.shape == (2, 2) else 1.0

        n0_pos = crosstab.loc[1, 0] if 1 in crosstab.index and 0 in crosstab.columns else 0
        n1_pos = crosstab.loc[1, 1] if 1 in crosstab.index and 1 in crosstab.columns else 0

        percent0 = (n0_pos / len(group0_cat) * 100) if len(group0_cat) > 0 else 0
        percent1 = (n1_pos / len(group1_cat) * 100) if len(group1_cat) > 0 else 0

        table_data.append({
            'Characteristic': name,
            f'{TARGET_COLUMN_FOR_LABELING} < {GROUP_THRESHOLD} (n={len(group0)})': f"{n0_pos} ({percent0:.1f})",
            f'{TARGET_COLUMN_FOR_LABELING} >= {GROUP_THRESHOLD} (n={len(group1)})': f"{n1_pos} ({percent1:.1f})",
            'P-Value': f"{p_val:.2f}" if p_val >= 0.01 else "<0.01"
        })

    return pd.DataFrame(table_data)


def create_split_characteristics_table(df: pd.DataFrame, train_indices, test_indices) -> pd.DataFrame:
    print("\n--- Supplementary Table 1: Train vs. Test Set Characteristics ---")

    df_train = df.loc[train_indices].drop_duplicates(subset=['Pt No']).copy()
    df_test = df.loc[test_indices].drop_duplicates(subset=['Pt No']).copy()

    table_data = []

    continuous_vars = ['age', 'POD', 'FV', 'RI', 'RA diameter', 'shunt diameter']
    for var in continuous_vars:
        if var not in df.columns:
            continue

        train_vals = df_train[var].dropna()
        test_vals = df_test[var].dropna()

        mean_train, std_train = train_vals.mean(), train_vals.std()
        mean_test, std_test = test_vals.mean(), test_vals.std()

        p_val = mannwhitneyu(train_vals, test_vals)[1] if len(train_vals) > 1 and len(test_vals) > 1 else np.nan

        table_data.append({
            'Characteristic': var,
            f'Training Set (n={len(df_train)})': f"{mean_train:.1f} ± {std_train:.1f}",
            f'Test Set (n={len(df_test)})': f"{mean_test:.1f} ± {std_test:.1f}",
            'P-Value': f"{p_val:.2f}" if not pd.isna(p_val) else "N/A"
        })

    categorical_vars = {
        'men': 'Male Sex (n, %)',
        'arterial calcification': 'Arterial Calcification (n, %)',
        'DM': 'DM (n, %)',
        'HTN': 'HTN (n, %)',
        'Heart disease': 'Heart Disease (n, %)',
        'tabaco site': 'tabaco site (n, %)',
        'left': 'left (n, %)'
    }

    mapping_dict = {'y': 1, 'n': 0, 'male': 1, 'female': 0, '1': 1, '0': 0, '1.0': 1, '0.0': 0}

    df_unique = df.drop_duplicates(subset=['Pt No']).copy()
    df_unique['set'] = np.where(df_unique.index.isin(train_indices), 'train', 'test')

    for var, name in categorical_vars.items():
        if var not in df.columns:
            continue

        df_unique[var + '_numeric'] = df_unique[var].astype(str).str.lower().map(mapping_dict)
        df_filtered = df_unique.dropna(subset=[var + '_numeric'])

        crosstab = pd.crosstab(df_filtered[var + '_numeric'], df_filtered['set'])

        p_val = chi2_contingency(crosstab)[1] if crosstab.shape[0] >= 2 and crosstab.shape[1] >= 2 else 1.0

        n_train_pos = crosstab.get('train', pd.Series(0)).get(1, 0)
        n_test_pos = crosstab.get('test', pd.Series(0)).get(1, 0)

        p_train = (n_train_pos / len(df_train) * 100) if len(df_train) > 0 else 0
        p_test = (n_test_pos / len(df_test) * 100) if len(df_test) > 0 else 0

        table_data.append({
            'Characteristic': name,
            f'Training Set (n={len(df_train)})': f"{n_train_pos} ({p_train:.1f})",
            f'Test Set (n={len(df_test)})': f"{n_test_pos} ({p_test:.1f})",
            'P-Value': f"{p_val:.2f}"
        })

    return pd.DataFrame(table_data)


def print_patient_ids_summary(df: pd.DataFrame, train_indices, test_indices, target_variable: str, output_dir: Path):
    print("\n" + "="*70)
    print("--- Patient ID Summary ---")
    print("="*70)

    df_train = df.loc[train_indices].drop_duplicates(subset=['Pt No'])
    train_class0 = df_train[df_train[target_variable] == 0]['Pt No'].sort_values().tolist()
    train_class1 = df_train[df_train[target_variable] == 1]['Pt No'].sort_values().tolist()

    print(f"\nTraining data:")
    print(f"  {TARGET_COLUMN_FOR_LABELING} < {GROUP_THRESHOLD} (n={len(train_class0)}): {train_class0}")
    print(f"  {TARGET_COLUMN_FOR_LABELING} >= {GROUP_THRESHOLD} (n={len(train_class1)}): {train_class1}")

    df_test = df.loc[test_indices].drop_duplicates(subset=['Pt No'])
    test_class0 = df_test[df_test[target_variable] == 0]['Pt No'].sort_values().tolist()
    test_class1 = df_test[df_test[target_variable] == 1]['Pt No'].sort_values().tolist()

    print(f"\nTest data:")
    print(f"  {TARGET_COLUMN_FOR_LABELING} < {GROUP_THRESHOLD} (n={len(test_class0)}): {test_class0}")
    print(f"  {TARGET_COLUMN_FOR_LABELING} >= {GROUP_THRESHOLD} (n={len(test_class1)}): {test_class1}")

    output_path = output_dir / "patient_ids_summary.csv"
    summary_df = pd.DataFrame({
        'Dataset': ['Train', 'Train', 'Test', 'Test'],
        'Class': [f'< {GROUP_THRESHOLD}', f'>= {GROUP_THRESHOLD}', f'< {GROUP_THRESHOLD}', f'>= {GROUP_THRESHOLD}'],
        'Count': [len(train_class0), len(train_class1), len(test_class0), len(test_class1)],
        'Patient_IDs': [', '.join(map(str, train_class0)), ', '.join(map(str, train_class1)),
                       ', '.join(map(str, test_class0)), ', '.join(map(str, test_class1))]
    })
    summary_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\nSaved patient IDs to '{output_path}'")
    print("="*70)

In [ ]:
# =========================================
# Feature Selection
# =========================================
def select_stable_features(X_train_df: pd.DataFrame, y_train: pd.Series,
                          groups_train: pd.Series, feature_list: list,
                          model_for_select, n_features_to_select: int,
                          n_bootstraps: int, random_state: int) -> list:
    print(f"-> Starting stability selection (bootstrap iterations: {n_bootstraps})...")

    feature_counter = Counter()
    unique_patients = groups_train.unique()
    n_patients = len(unique_patients)
    rng = np.random.RandomState(random_state)

    if isinstance(model_for_select, Pipeline):
        scaler = clone(model_for_select.steps[0][1])
        estimator = clone(model_for_select.steps[-1][1])
    else:
        scaler = None
        estimator = clone(model_for_select)

    selector = RFE(estimator=estimator, n_features_to_select=n_features_to_select, step=1)
    X_train_subset = X_train_df[feature_list]

    valid_iterations = 0
    for i in range(n_bootstraps):
        if (i + 1) % 100 == 0:
            print(f"  ... Bootstrap {i+1}/{n_bootstraps} complete")

        try:
            sampled_patients = rng.choice(unique_patients, size=n_patients, replace=True)
            mask = groups_train.isin(sampled_patients)
            idx = groups_train[mask].index

            if len(idx) == 0 or len(np.unique(y_train.loc[idx])) < 2:
                continue

            X_boot = X_train_subset.loc[idx]
            y_boot = y_train.loc[idx]

            X_boot_scaled = scaler.fit_transform(X_boot) if scaler else X_boot

            selector.fit(X_boot_scaled, y_boot)
            selected = X_boot.columns[selector.support_].tolist()

            feature_counter.update(selected)
            valid_iterations += 1

        except Exception:
            continue

    if valid_iterations == 0:
        print("Warning: No valid bootstrap iterations.")
        return feature_list[:n_features_to_select]

    most_common = [f for f, _ in feature_counter.most_common(n_features_to_select)]

    print(f"-> Stability selection complete. ({valid_iterations}/{n_bootstraps} valid iterations)")
    print("--- Selected features (by frequency) ---")
    for feature, count in feature_counter.most_common(10):
        print(f"  {feature}: {count} ({count/valid_iterations:.1%})")

    return most_common

import hashlib

def _cache_path(output_dir, task, phase, pool_name):
    """Generate cache file path for feature selection results."""
    return output_dir / f"feature_selection_cache_{task}_{phase}_{pool_name}.json"


def select_stable_features_cached(df_train, y_train, groups_train, feature_list,
                                   model_for_select, n_features, n_bootstraps,
                                   random_state, output_dir, task_label, phase, pool_name):
    """Wrapper: load from cache if exists, otherwise run and save."""
    import json as _json
    cache_file = _cache_path(output_dir, task_label, phase, pool_name)

    if not FORCE_RESELECT and cache_file.exists():
        with open(cache_file, 'r') as f:
            cached = _json.load(f)
        print(f"  [CACHE HIT] Loaded from {cache_file.name}")
        print(f"  Selected features: {cached['selected_features']}")
        return cached['selected_features']

    # Run fresh
    print(f"  [RUNNING] Stability selection ({n_bootstraps} bootstraps)...")
    selected_cols = select_stable_features(
        df_train, y_train, groups_train, feature_list,
        model_for_select, n_features, n_bootstraps, random_state
    )

    # Save cache
    output_dir.mkdir(exist_ok=True)
    cache_data = {
        'selected_features': selected_cols,
        'n_bootstraps': n_bootstraps,
        'random_state': random_state,
        'pool_name': pool_name,
        'n_features': n_features
    }
    with open(cache_file, 'w') as f:
        _json.dump(cache_data, f, indent=2)
    print(f"  [SAVED] Cache saved to {cache_file.name}")

    return selected_cols


In [ ]:
# ========================================
# Model Evaluation
# ========================================
def evaluate_model_with_cv(X_selected: pd.DataFrame, y: pd.Series,
                          groups: pd.Series, model) -> pd.DataFrame:
    group_kfold = GroupKFold(n_splits=N_SPLITS_CV)
    scores = {
        'auc': [], 'accuracy': [], 'sensitivity': [],
        'specificity': [], 'f1_score': []
    }

    for train_idx, test_idx in group_kfold.split(X_selected, y, groups):
        X_train_cv = X_selected.iloc[train_idx]
        X_test_cv = X_selected.iloc[test_idx]
        y_train_cv = y.iloc[train_idx]
        y_test_cv = y.iloc[test_idx]

        model_clone = clone(model)
        model_clone.fit(X_train_cv, y_train_cv)

        y_pred_proba = model_clone.predict_proba(X_test_cv)[:, 1]
        y_pred = model_clone.predict(X_test_cv)

        if len(np.unique(y_test_cv)) > 1:
            scores['auc'].append(roc_auc_score(y_test_cv, y_pred_proba))
        else:
            scores['auc'].append(np.nan)

        report = classification_report(y_test_cv, y_pred, output_dict=True, zero_division=0)
        scores['accuracy'].append(report['accuracy'])
        scores['sensitivity'].append(report.get('1', {}).get('recall', np.nan))
        scores['specificity'].append(report.get('0', {}).get('recall', np.nan))
        scores['f1_score'].append(report['weighted avg']['f1-score'])

    return pd.DataFrame(scores).agg(['mean', 'std']).T


def calculate_bootstrap_ci(y_test: pd.Series, y_pred_proba: np.ndarray,
                          test_patient_ids: pd.Series) -> tuple:
    unique_patients = test_patient_ids.unique()
    n_patients = len(unique_patients)
    rng = np.random.RandomState(UNIFIED_RANDOM_STATE)

    bootstrapped_scores = []
    for _ in range(N_BOOTSTRAPS_CI):
        sampled_patients = rng.choice(unique_patients, size=n_patients, replace=True)
        mask = test_patient_ids.isin(sampled_patients)
        idx = test_patient_ids[mask].index

        if len(idx) == 0:
            continue

        y_true_bs = y_test.loc[idx]
        idx_positions = y_test.index.get_indexer(idx)
        y_pred_bs = y_pred_proba[idx_positions]

        if len(np.unique(y_true_bs)) < 2:
            continue

        bootstrapped_scores.append(roc_auc_score(y_true_bs, y_pred_bs))

    if not bootstrapped_scores:
        return np.nan, np.nan

    return np.percentile(bootstrapped_scores, 2.5), np.percentile(bootstrapped_scores, 97.5)


def get_feature_importances(model, columns: list) -> pd.Series:
    estimator = model.steps[-1][1] if isinstance(model, Pipeline) else model

    if hasattr(estimator, 'feature_importances_'):
        importances = estimator.feature_importances_
    elif hasattr(estimator, 'coef_'):
        importances = np.abs(estimator.coef_[0])
    else:
        importances = np.zeros(len(columns))

    return pd.Series(importances, index=columns).sort_values(ascending=False)


def display_test_auc_summary(final_results: dict, y_pred_probas_test: dict,
                             y_test_global: pd.Series, test_patient_ids_global: pd.Series,
                             phase_name: str = "Phase 1"):
    print("\n" + "="*70)
    print(f"### {phase_name}: Test AUC Summary ###")
    print("="*70)

    results = []
    for model_id in final_results.keys():
        if model_id not in y_pred_probas_test:
            continue

        proba = y_pred_probas_test[model_id].values
        test_auc = roc_auc_score(y_test_global, proba)
        ci_lower, ci_upper = calculate_bootstrap_ci(y_test_global, proba, test_patient_ids_global)

        results.append({
            'Model': model_id,
            'Test_AUC': test_auc,
            'CI_Lower': ci_lower,
            'CI_Upper': ci_upper
        })

        print(f"\n{model_id}:")
        print(f"  Test AUC: {test_auc:.4f} [{ci_lower:.4f}-{ci_upper:.4f}]")

    if results:
        results_df = pd.DataFrame(results)
        print("\n" + "="*70)
        print("Formatted for Table:")
        print("="*70)
        for _, row in results_df.iterrows():
            print(f"{row['Model']:30s}: Test AUC = {row['Test_AUC']:.3f} [{row['CI_Lower']:.3f}-{row['CI_Upper']:.3f}]")

    print("="*70)
    return results

# =========================================== ####
# Hyperparameter Tuning Functions
# =========================================== ####

def tune_rf_with_oob(X_train, y_train, param_grid, random_state):
    """Tune RF using OOB-AUC (no additional CV loop)."""
    best_score = -1
    best_params = None
    best_model = None

    param_combinations = list(iter_product(*param_grid.values()))
    param_names = list(param_grid.keys())

    for combo in param_combinations:
        params = dict(zip(param_names, combo))
        rf = RandomForestClassifier(
            **params,
            random_state=random_state,
            class_weight='balanced',
            oob_score=True
        )
        rf.fit(X_train, y_train)
        oob_auc = roc_auc_score(y_train, rf.oob_decision_function_[:, 1])

        if oob_auc > best_score:
            best_score = oob_auc
            best_params = params
            best_model = rf

    return best_model, best_params, best_score


def tune_lr_with_gridsearch(X_train, y_train, groups_train, param_grid, random_state):
    """Tune LR using GridSearchCV + Pipeline + GroupKFold (no scaling leak)."""
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            class_weight='balanced',
            random_state=random_state,
            max_iter=1000
        ))
    ])
    inner_cv = GroupKFold(n_splits=3)
    grid_search = GridSearchCV(
        pipeline, param_grid=param_grid,
        cv=inner_cv, scoring='roc_auc', refit=True
    )
    grid_search.fit(X_train, y_train, groups=groups_train)
    return grid_search.best_estimator_, grid_search.best_params_['model__C']




# ============================================================
# Nested CV: Feature Selection Once, Evaluate Selected Models
# ============================================================

def nested_cv_both_models(X_all_features, y, groups,
                          feature_list, n_features_to_select,
                          n_bootstraps_stability, random_state,
                          models_to_run=None):
    """
    Nested CV using StratifiedGroupKFold.
    Feature selection ONCE per fold, then evaluate models in models_to_run.
    """
    if models_to_run is None:
        models_to_run = MODELS_TO_RUN

    outer_cv = StratifiedGroupKFold(
        n_splits=N_SPLITS_CV, shuffle=True, random_state=random_state
    )

    rf_for_selection = RandomForestClassifier(
        random_state=random_state, class_weight='balanced'
    )

    all_scores = {}
    all_fold_details = {}
    for mt in models_to_run:
        all_scores[mt] = {'auc': [], 'accuracy': [], 'sensitivity': [], 'specificity': [], 'f1_score': []}
        all_fold_details[mt] = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_all_features, y, groups)):
        fold_start = time.time()

        X_train_cv = X_all_features.iloc[train_idx].copy()
        X_test_cv = X_all_features.iloc[test_idx].copy()

        # Impute missing values using fold training data only
        for col in X_train_cv.columns[X_train_cv.isnull().any()]:
            fill_val = X_train_cv[col].mean()
            X_train_cv[col] = X_train_cv[col].fillna(fill_val)
            X_test_cv[col] = X_test_cv[col].fillna(fill_val)

        y_train_cv = y.iloc[train_idx]
        y_test_cv = y.iloc[test_idx]
        groups_train_cv = groups.iloc[train_idx]

        # Step 1: Feature selection ONCE per fold
        selected_cols = select_stable_features(
            X_train_cv, y_train_cv, groups_train_cv, feature_list,
            rf_for_selection, n_features_to_select,
            n_bootstraps_stability, random_state
        )

        X_train_sel = X_train_cv[selected_cols]
        X_test_sel = X_test_cv[selected_cols]

        fs_time = time.time() - fold_start
        print(f"    Fold {fold_idx+1}/{N_SPLITS_CV}: Features={selected_cols} ({fs_time:.0f}s)")

        # Step 2: Evaluate selected models
        if 'RF' in models_to_run:
            rf_model, rf_params, rf_oob = tune_rf_with_oob(
                X_train_sel, y_train_cv, RF_PARAM_GRID, random_state
            )
            rf_hp = f"n_est={rf_params['n_estimators']}, depth={rf_params['max_depth']}, leaf={rf_params['min_samples_leaf']}, OOB-AUC={rf_oob:.3f}"

            rf_proba = rf_model.predict_proba(X_test_sel)[:, 1]
            rf_pred = rf_model.predict(X_test_sel)
            rf_auc = roc_auc_score(y_test_cv, rf_proba) if len(np.unique(y_test_cv)) > 1 else np.nan

            rf_report = classification_report(y_test_cv, rf_pred, output_dict=True, zero_division=0)
            all_scores['RF']['auc'].append(rf_auc)
            all_scores['RF']['accuracy'].append(rf_report['accuracy'])
            all_scores['RF']['sensitivity'].append(rf_report.get('1', {}).get('recall', np.nan))
            all_scores['RF']['specificity'].append(rf_report.get('0', {}).get('recall', np.nan))
            all_scores['RF']['f1_score'].append(rf_report['weighted avg']['f1-score'])
            all_fold_details['RF'].append({'fold': fold_idx+1, 'features': selected_cols, 'hp': rf_hp, 'auc': rf_auc})
            print(f"      RF: AUC={rf_auc:.3f}, HP=[{rf_hp}]")

        if 'LR' in models_to_run:
            lr_model, lr_C = tune_lr_with_gridsearch(
                X_train_sel, y_train_cv, groups_train_cv, LR_PARAM_GRID, random_state
            )
            lr_hp = f"C={lr_C}"

            lr_proba = lr_model.predict_proba(X_test_sel)[:, 1]
            lr_pred = lr_model.predict(X_test_sel)
            lr_auc = roc_auc_score(y_test_cv, lr_proba) if len(np.unique(y_test_cv)) > 1 else np.nan

            lr_report = classification_report(y_test_cv, lr_pred, output_dict=True, zero_division=0)
            all_scores['LR']['auc'].append(lr_auc)
            all_scores['LR']['accuracy'].append(lr_report['accuracy'])
            all_scores['LR']['sensitivity'].append(lr_report.get('1', {}).get('recall', np.nan))
            all_scores['LR']['specificity'].append(lr_report.get('0', {}).get('recall', np.nan))
            all_scores['LR']['f1_score'].append(lr_report['weighted avg']['f1-score'])
            all_fold_details['LR'].append({'fold': fold_idx+1, 'features': selected_cols, 'hp': lr_hp, 'auc': lr_auc})

            fold_time = time.time() - fold_start
            print(f"      LR: AUC={lr_auc:.3f}, HP=[{lr_hp}] (fold total: {fold_time:.0f}s)")

    results = {}
    for mt in models_to_run:
        results[mt] = {
            'perf': pd.DataFrame(all_scores[mt]).agg(['mean', 'std']).T,
            'details': all_fold_details[mt]
        }
    return results


In [ ]:
# ===============================
# AUC Comparison (DeLong Test)
# ===============================
def compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T + 1
    return T2


def fast_delong_auc_cov(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    m = sum(y_true == 1)
    n = sum(y_true == 0)

    v_10 = compute_midrank(y_pred[y_true == 1])
    v_01 = compute_midrank(y_pred[y_true == 0])

    auc = (np.sum(v_10) - m * (m + 1) / 2) / (m * n)

    s_10 = v_10 / m - (m + 1) / (2 * m)
    s_01 = v_01 / n - (n + 1) / (2 * n)

    var_10 = (np.sum(s_10**2) - m * (s_10.sum()/m)**2) / (m - 1)
    var_01 = (np.sum(s_01**2) - n * (s_01.sum()/n)**2) / (n - 1)

    return auc, var_10, var_01


def delong_roc_test(y_true, scores_a, scores_b):
    y_true = np.asarray(y_true)
    scores_a = np.asarray(scores_a)
    scores_b = np.asarray(scores_b)

    m = sum(y_true == 1)
    n = sum(y_true == 0)

    if m == 0 or n == 0:
        return 1.0

    auc_a, var_a_10, var_a_01 = fast_delong_auc_cov(y_true, scores_a)
    auc_b, var_b_10, var_b_01 = fast_delong_auc_cov(y_true, scores_b)

    v_a_10 = compute_midrank(scores_a[y_true == 1])
    v_a_01 = compute_midrank(scores_a[y_true == 0])
    v_b_10 = compute_midrank(scores_b[y_true == 1])
    v_b_01 = compute_midrank(scores_b[y_true == 0])

    s_a_10 = v_a_10 / m - (m + 1) / (2 * m)
    s_a_01 = v_a_01 / n - (n + 1) / (2 * n)
    s_b_10 = v_b_10 / m - (m + 1) / (2 * m)
    s_b_01 = v_b_01 / n - (n + 1) / (2 * n)

    cov_10 = (np.sum(s_a_10 * s_b_10) - m * (s_a_10.sum()/m) * (s_b_10.sum()/m)) / (m - 1)
    cov_01 = (np.sum(s_a_01 * s_b_01) - n * (s_a_01.sum()/n) * (s_b_01.sum()/n)) / (n - 1)

    var_diff = var_a_10 / m + var_a_01 / n + var_b_10 / m + var_b_01 / n - 2 * (cov_10 / m + cov_01 / n)

    if var_diff <= 1e-8:
        return 1.0

    z = (auc_a - auc_b) / np.sqrt(var_diff)
    p = 2 * norm.sf(np.abs(z))

    return p


def compute_auc_diff_paired(y_test: pd.Series, y_pred_probas: pd.DataFrame,
                           test_patient_ids: pd.Series, model_pair: tuple) -> dict:
    model_a, model_b = model_pair
    proba_a = y_pred_probas[model_a]
    proba_b = y_pred_probas[model_b]

    unique_patients = test_patient_ids.unique()
    n_patients = len(unique_patients)
    rng = np.random.RandomState(UNIFIED_RANDOM_STATE)

    auc_diffs = []
    for _ in range(N_BOOTSTRAPS_CI):
        sampled_patients = rng.choice(unique_patients, size=n_patients, replace=True)
        mask = test_patient_ids.isin(sampled_patients)
        idx = test_patient_ids[mask].index

        if len(idx) == 0:
            continue

        y_true_bs = y_test.loc[idx]

        if len(np.unique(y_true_bs)) < 2:
            continue

        auc_a = roc_auc_score(y_true_bs, proba_a.loc[idx])
        auc_b = roc_auc_score(y_true_bs, proba_b.loc[idx])
        auc_diffs.append(auc_a - auc_b)

    if not auc_diffs:
        return {
            'model_pair': f"{model_a} vs {model_b}",
            'auc_diff_mean': np.nan,
            'ci_lower': np.nan,
            'ci_upper': np.nan,
            'p_value': np.nan
        }

    auc_diffs = np.array(auc_diffs)
    mean_diff = auc_diffs.mean()
    ci_lower = np.percentile(auc_diffs, 2.5)
    ci_upper = np.percentile(auc_diffs, 97.5)

    p_val = min(np.mean(auc_diffs <= 0), np.mean(auc_diffs >= 0)) * 2
    p_val = min(p_val, 1.0)

    return {
        'model_pair': f"{model_a} vs {model_b}",
        'auc_diff_mean': mean_diff,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'p_value': p_val
    }

In [ ]:
# =====================================
# Visualization
# =====================================
def analyze_and_plot_shap(model, X_train, X_test, output_dir, model_name, target_name):
    import shap

    print("\n" + "="*70)
    print("--- SHAP Analysis ---")
    print("="*70)

    plt.rcParams['font.family'] = 'Liberation Sans'

    model_to_explain = model.steps[-1][1] if isinstance(model, Pipeline) else model
    print(f"Initializing SHAP explainer for: {type(model_to_explain).__name__}")

    explainer = shap.Explainer(model_to_explain, X_train)
    explanation = explainer(X_test)

    explanation_plot = explanation[:, :, 1] if hasattr(explanation, 'values') and explanation.values.ndim == 3 else explanation

    print("Generating SHAP Summary Plot...")
    plt.figure(figsize=(10, 8))
    shap.summary_plot(explanation_plot, features=X_test, show=False, plot_size=None)
    fig_summary = output_dir / f'SHAP_Summary_{target_name}.svg'
    plt.title(f'SHAP Summary Plot ({model_name})', fontsize=16, weight='bold')
    plt.tight_layout()
    plt.savefig(fig_summary, format='svg', bbox_inches='tight')
    plt.show()
    print(f"Saved to '{fig_summary}'")

    return [fig_summary]


def plot_calibration_curve(model, X_test, y_test, model_name, output_dir, target_name):
    print("\n" + "="*70)
    print("--- Calibration Analysis ---")
    print("="*70)

    y_pred_proba = model.predict_proba(X_test)[:, 1]
    brier = brier_score_loss(y_test, y_pred_proba)

    print(f"Brier Score for '{model_name}': {brier:.4f}")
    print("(Brier score closer to 0 indicates better calibration)")

    fig, ax = plt.subplots(figsize=(10, 8))
    CalibrationDisplay.from_predictions(
        y_test, y_pred_proba, n_bins=5, name=model_name,
        ax=ax, strategy='uniform'
    )

    ax.set_title(f'Calibration Plot ({model_name})', fontsize=16, weight='bold')
    ax.set_xlabel("Mean Predicted Probability", fontsize=12, weight='bold')
    ax.set_ylabel("Fraction of Positives", fontsize=12, weight='bold')
    ax.grid(linestyle=':', alpha=0.6)

    fig_path = output_dir / f'Calibration_{target_name}.svg'
    plt.tight_layout()
    plt.savefig(fig_path, format='svg', bbox_inches='tight')
    plt.show()

    print(f"Saved to '{fig_path}'")
    return fig_path


def plot_roc_curves(final_results: dict, table2_df: pd.DataFrame, output_file: Path):
    plt.figure(figsize=(12, 10))
    ax = plt.gca()

    auc_col = 'AUC' if 'AUC' in table2_df.columns else 'AUC (95% CI)'

    for model_id in table2_df['Model'].unique():
        if model_id not in final_results:
            continue

        result = final_results[model_id]
        auc_text = table2_df.loc[table2_df['Model'] == model_id, auc_col].values[0]
        if auc_col == 'AUC':
            cv_auc = auc_text.split(' ±')[0]
        else:
            cv_auc = auc_text.split(' -')[0]
        label = f"{model_id} (AUC = {float(cv_auc):.3f})"

        RocCurveDisplay.from_estimator(
            result['model'], result['X_test'], result['y_test'],
            name=label, ax=ax
        )

    plt.plot([0, 1], [0, 1], color='black', lw=1, linestyle='--')
    plt.title('ROC Curves', fontsize=16, weight='bold')
    plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12, weight='bold')
    plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12, weight='bold')
    plt.grid(linestyle=':', alpha=0.6)
    plt.legend(fontsize=10, loc='lower right')
    plt.tight_layout()
    plt.savefig(output_file, format='svg', bbox_inches='tight')
    plt.show()

    print(f"Saved to '{output_file}'")


def save_tables_to_word(table1_df, supp_table1_df, table2_df, table3_df, table4_data, output_file, target_variable):
    doc = Document()

    if table1_df is not None and not table1_df.empty:
        doc.add_heading(f'Table 1: Patient Characteristics ({target_variable})', level=1)
        t1 = doc.add_table(table1_df.shape[0] + 1, table1_df.shape[1], style='Table Grid')
        for j, col in enumerate(table1_df.columns):
            t1.cell(0, j).text = col
        for i in range(table1_df.shape[0]):
            for j in range(table1_df.shape[1]):
                t1.cell(i + 1, j).text = str(table1_df.iloc[i, j])
        doc.add_page_break()

    if supp_table1_df is not None and not supp_table1_df.empty:
        doc.add_heading('Supplementary Table 1: Train vs. Test Set Characteristics', level=1)
        ts1 = doc.add_table(supp_table1_df.shape[0] + 1, supp_table1_df.shape[1], style='Table Grid')
        for j, col in enumerate(supp_table1_df.columns):
            ts1.cell(0, j).text = col
        for i in range(supp_table1_df.shape[0]):
            for j in range(supp_table1_df.shape[1]):
                ts1.cell(i + 1, j).text = str(supp_table1_df.iloc[i, j])
        doc.add_page_break()

    doc.add_heading(f'Table 2: Model Performance Summary ({target_variable})', level=1)
    t2 = doc.add_table(table2_df.shape[0] + 1, table2_df.shape[1], style='Table Grid')
    for j, col in enumerate(table2_df.columns):
        t2.cell(0, j).text = col
    for i in range(table2_df.shape[0]):
        for j in range(table2_df.shape[1]):
            t2.cell(i + 1, j).text = str(table2_df.iloc[i, j])
    doc.add_page_break()

    if table3_df is not None and not table3_df.empty:
        doc.add_heading(f'Table 3: Feature Importances ({target_variable})', level=1)
        table3_reset = table3_df.reset_index()
        t3 = doc.add_table(table3_reset.shape[0] + 1, table3_reset.shape[1], style='Table Grid')
        for j, col in enumerate(table3_reset.columns):
            t3.cell(0, j).text = col
        for i in range(table3_reset.shape[0]):
            for j in range(table3_reset.shape[1]):
                val = table3_reset.iloc[i, j]
                t3.cell(i + 1, j).text = f"{val:.4f}" if isinstance(val, (float, np.floating)) else str(val)
        doc.add_page_break()

    doc.add_heading('Table 4: AUC Difference Comparisons', level=1)
    if not table4_data:
        doc.add_paragraph("No AUC difference comparisons were performed.")
    else:
        for idx, (prefix, df) in enumerate(table4_data.items()):
            sub_label = chr(97 + idx)
            doc.add_heading(f'Table 4{sub_label}: {prefix} Model Comparisons', level=2)
            if df.empty:
                doc.add_paragraph("No data available.")
                continue
            t4 = doc.add_table(df.shape[0] + 1, df.shape[1], style='Table Grid')
            for j, col in enumerate(df.columns):
                t4.cell(0, j).text = col
            for i in range(df.shape[0]):
                for j in range(df.shape[1]):
                    t4.cell(i + 1, j).text = str(df.iloc[i, j])
            doc.add_paragraph()

    doc.save(output_file)
    print(f"\nSaved tables to '{output_file}'")


In [ ]:
# ===============================================
# Phase 1: Engineered vs Perceptual Features
# ===============================================
def main_phase1():
    print("\n" + "="*80)
    print("### STARTING PHASE 1 ANALYSIS ###")
    print(f"### Models: {MODELS_TO_RUN} | Mode: {RUN_MODE} ###")
    print("="*80)

    OUTPUT_DIR.mkdir(exist_ok=True)

    TARGET_VARIABLE, df_orig = load_and_preprocess_data(
        INPUT_EXCEL_PATH, TARGET_COLUMN_FOR_LABELING, GROUP_THRESHOLD
    )

    table1_df = create_patient_characteristics_table(df_orig, TARGET_VARIABLE)
    print(table1_df.to_string())

    df_processed = df_orig.copy()
    y = df_processed[TARGET_VARIABLE]
    groups = df_processed['Pt No']

    feature_sets = {
        "A-D": [col for col in df_processed.columns if re.match(r"^[A-D]\d+", col)],
        "E_only": [col for col in df_processed.columns if re.match(r"^E\d+", col)],
    }

    numeric_cols = list(set([f for fs in feature_sets.values() for f in fs]))
    for col in numeric_cols:
        df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce').fillna(
            df_processed[col].mean()
        )

    train_indices, test_indices = split_data_by_patient(
        df_processed, TARGET_VARIABLE, test_size=0.2, random_state=UNIFIED_RANDOM_STATE
    )

    supp_table1_df = create_split_characteristics_table(df_orig, train_indices, test_indices)
    print(supp_table1_df.to_string())

    print_patient_ids_summary(df_processed, train_indices, test_indices, TARGET_VARIABLE, OUTPUT_DIR)

    # --- Feature selection (500 bootstraps) ---
    print("\n--- Selecting stable features using training data only ---")
    df_train = df_processed.loc[train_indices]
    y_train = y.loc[train_indices]
    groups_train = groups.loc[train_indices]

    rf_selector = RandomForestClassifier(
        random_state=UNIFIED_RANDOM_STATE, class_weight='balanced'
    )

    selected_feature_dfs = {}
    for name, f_list in feature_sets.items():
        print(f"\nFeature set '{name}':")
        selected_cols = select_stable_features_cached(
            df_train, y_train, groups_train, f_list,
            rf_selector, N_FEATURES_TO_SELECT,
            N_BOOTSTRAPS_STABILITY, UNIFIED_RANDOM_STATE,
            OUTPUT_DIR, f"{TARGET_COLUMN_FOR_LABELING}_{GROUP_THRESHOLD}",
            "Phase1", name
        )
        selected_feature_dfs[name] = df_processed[selected_cols]

    # === Nested CV ===
    nested_cv_results = {}
    if RUN_MODE in ['full', 'cv_only']:
        print("\n--- Nested CV with HP Tuning ---")
        for pool_name, f_list_full in feature_sets.items():
            print(f"\n  Pool: {pool_name}")
            pool_results = nested_cv_both_models(
                X_all_features=df_train[f_list_full],
                y=y_train, groups=groups_train,
                feature_list=f_list_full,
                n_features_to_select=N_FEATURES_TO_SELECT,
                n_bootstraps_stability=N_BOOTSTRAPS_STABILITY,
                random_state=UNIFIED_RANDOM_STATE
            )
            for mt in MODELS_TO_RUN:
                nested_cv_results[f"{mt}_{pool_name}"] = pool_results[mt]

        # Print CV summary
        print("\n" + "="*50)
        print("--- Nested CV Results ---")
        print("="*50)
        for label, res in nested_cv_results.items():
            perf = res['perf']
            print(f"{label}: CV-AUC={perf.loc['auc','mean']:.3f}±{perf.loc['auc','std']:.3f}, "
                  f"Acc={perf.loc['accuracy','mean']:.3f}±{perf.loc['accuracy','std']:.3f}, "
                  f"Sens={perf.loc['sensitivity','mean']:.3f}±{perf.loc['sensitivity','std']:.3f}, "
                  f"Spec={perf.loc['specificity','mean']:.3f}±{perf.loc['specificity','std']:.3f}")

        print("\n--- Per-Fold Details ---")
        for label, res in nested_cv_results.items():
            print(f"\n  {label}:")
            for d in res['details']:
                print(f"    Fold {d['fold']}: AUC={d['auc']:.3f}, HP=[{d['hp']}], Features={d['features']}")

    if RUN_MODE == 'cv_only':
        print("\n### PHASE 1 CV-ONLY COMPLETE ###")
        return TARGET_VARIABLE, df_orig, train_indices, test_indices

    # === Test evaluation ===
    if RUN_MODE in ['full', 'test_only']:
        print("\n--- Test Evaluation with HP-tuned Models ---")
        final_results = {}
        summary_data = []
        table3_data = []
        y_pred_probas_test = {}
        hp_summary = []

        y_test_global = y.loc[test_indices]
        test_patient_ids_global = groups.loc[test_indices]

        for feature_set_name, X_selected in selected_feature_dfs.items():
            X_train = X_selected.loc[train_indices]
            X_test = X_selected.loc[test_indices]
            y_train_loop = y.loc[train_indices]
            y_test_loop = y.loc[test_indices]
            groups_test = groups.loc[test_indices]

            for model_type in MODELS_TO_RUN:
                model_id = f"{model_type}_{feature_set_name}"
                print(f"\nProcessing: {model_id}")

                if model_type == 'RF':
                    final_model, best_params, oob = tune_rf_with_oob(
                        X_train, y_train_loop, RF_PARAM_GRID, UNIFIED_RANDOM_STATE
                    )
                    hp_info = f"n_est={best_params['n_estimators']}, depth={best_params['max_depth']}, leaf={best_params['min_samples_leaf']}, OOB-AUC={oob:.3f}"
                else:
                    final_model, best_C = tune_lr_with_gridsearch(
                        X_train, y_train_loop, groups_train, LR_PARAM_GRID, UNIFIED_RANDOM_STATE
                    )
                    hp_info = f"C={best_C}"

                print(f"  Best HP: {hp_info}")
                hp_summary.append({'Model': model_id, 'Best HP': hp_info})

                proba_test = final_model.predict_proba(X_test)[:, 1]
                y_pred_probas_test[model_id] = pd.Series(proba_test, index=X_test.index)

                ci_lower, ci_upper = calculate_bootstrap_ci(y_test_loop, proba_test, groups_test)

                row = {
                    'Model': model_id,
                    'Features': X_selected.shape[1],
                    'AUC (95% CI)': f"{ci_lower:.3f} - {ci_upper:.3f}",
                }
                if model_id in nested_cv_results:
                    cv_perf = nested_cv_results[model_id]['perf']
                    row['CV-AUC'] = f"{cv_perf.loc['auc', 'mean']:.3f} ± {cv_perf.loc['auc', 'std']:.3f}"
                    row['CV-Accuracy'] = f"{cv_perf.loc['accuracy', 'mean']:.3f} ± {cv_perf.loc['accuracy', 'std']:.3f}"
                    row['CV-Sensitivity'] = f"{cv_perf.loc['sensitivity', 'mean']:.3f} ± {cv_perf.loc['sensitivity', 'std']:.3f}"
                    row['CV-Specificity'] = f"{cv_perf.loc['specificity', 'mean']:.3f} ± {cv_perf.loc['specificity', 'std']:.3f}"
                    row['CV-F1'] = f"{cv_perf.loc['f1_score', 'mean']:.3f} ± {cv_perf.loc['f1_score', 'std']:.3f}"
                summary_data.append(row)

                final_results[model_id] = {
                    'model': final_model,
                    'X_test': X_test,
                    'y_test': y_test_loop,
                    'X_train': X_train
                }

                importances = get_feature_importances(final_model, X_selected.columns)
                for feature, importance in importances.items():
                    table3_data.append({
                        'Model_ID': model_id, 'Feature': feature, 'Importance': importance
                    })

        table2_df = pd.DataFrame(summary_data)
        table3_df = pd.DataFrame(table3_data).pivot_table(
            index='Feature', columns='Model_ID', values='Importance'
        ).sort_index()

        print("\n" + "="*50)
        print("--- Table 2: Model Performance Summary ---")
        print("="*50)
        print(table2_df.to_string(index=False))

        print("\n" + "="*50)
        print("--- Table 3: Top Features per Model ---")
        print("="*50)
        print(table3_df)

        print("\n--- Best Hyperparameters ---")
        print(pd.DataFrame(hp_summary).to_string(index=False))

        display_test_auc_summary(
            final_results, y_pred_probas_test,
            y_test_global, test_patient_ids_global, "Phase 1"
        )

        # SHAP + Calibration
        shap_model = "RF_E_only" if "RF_E_only" in final_results else list(final_results.keys())[0]
        print(f"\n--- Detailed Analysis: {shap_model} (Phase 1) ---")
        if shap_model in final_results:
            results = final_results[shap_model]
            analyze_and_plot_shap(
                results['model'], results['X_train'], results['X_test'],
                OUTPUT_DIR, shap_model, f"{TARGET_COLUMN_FOR_LABELING}_Phase1"
            )
            plot_calibration_curve(
                results['model'], results['X_test'], results['y_test'],
                shap_model, OUTPUT_DIR, f"{TARGET_COLUMN_FOR_LABELING}_Phase1"
            )

        # Table 4: AUC Difference (only if both pools have same model type)
        y_pred_probas_df = pd.DataFrame(y_pred_probas_test)
        table4_subtables = {}

        for prefix in MODELS_TO_RUN:
            pair = (f'{prefix}_E_only', f'{prefix}_A-D')
            if all(p in y_pred_probas_df.columns for p in pair):
                res = compute_auc_diff_paired(y_test_global, y_pred_probas_df, test_patient_ids_global, pair)
                p_delong = delong_roc_test(
                    y_test_global.values,
                    y_pred_probas_df[pair[0]].values,
                    y_pred_probas_df[pair[1]].values
                )
                res['p_value (DeLong)'] = p_delong
                sub_df = pd.DataFrame([res])
                pvals_bs = sub_df['p_value'].dropna()
                if not pvals_bs.empty:
                    _, pvals_fdr, _, _ = multipletests(pvals_bs, alpha=0.05, method='fdr_bh')
                    sub_df['p-value (Bootstrap, FDR)'] = pvals_fdr
                pvals_delong = sub_df['p_value (DeLong)'].dropna()
                if not pvals_delong.empty:
                    _, pvals_fdr, _, _ = multipletests(pvals_delong, alpha=0.05, method='fdr_bh')
                    sub_df['p-value (DeLong, FDR)'] = pvals_fdr
                table4_subtables[prefix] = sub_df

        if table4_subtables:
            print("\n" + "="*70)
            print("--- Table 4: AUC Difference (Test Data) ---")
            print("="*70)
            for prefix, df in table4_subtables.items():
                print(f"\n--- {prefix} Model Comparisons ---")
                print(df.to_string(index=False))

        save_tables_to_word(
            table1_df, supp_table1_df, table2_df, table3_df, table4_subtables,
            OUTPUT_DIR / f'Tables_{TARGET_COLUMN_FOR_LABELING}_Phase1.docx',
            TARGET_VARIABLE
        )

        plot_roc_curves(
            final_results, table2_df,
            OUTPUT_DIR / f'ROC_Curve_{TARGET_COLUMN_FOR_LABELING}_Phase1.svg'
        )

    print("\n" + "="*80)
    print("### PHASE 1 ANALYSIS COMPLETE ###")
    print("="*80)

    return TARGET_VARIABLE, df_orig, train_indices, test_indices


In [ ]:
# ===============================================
# Phase 2: Acoustic Only vs Acoustic + Clinical
# ===============================================
def main_phase2():
    print("\n" + "="*80)
    print("### STARTING PHASE 2 ANALYSIS ###")
    print(f"### Models: {MODELS_TO_RUN} | Mode: {RUN_MODE} ###")
    print("="*80)

    OUTPUT_DIR_P2 = DRIVE_BASE_PATH / f"{TARGET_COLUMN_FOR_LABELING}_{GROUP_THRESHOLD}_results_phase2"
    OUTPUT_DIR_P2.mkdir(exist_ok=True)

    TARGET_VARIABLE, df_orig = load_and_preprocess_data(
        INPUT_EXCEL_PATH, TARGET_COLUMN_FOR_LABELING, GROUP_THRESHOLD
    )

    df_processed = df_orig.copy()

    binary_cols = ['men', 'arterial calcification', 'DM', 'HTN', 'Heart disease', 'tabaco site', 'left']
    mapping_dict = {'y': 1, 'n': 0, 'male': 1, 'female': 0, '1': 1, '0': 0, '1.0': 1, '0.0': 0}
    numeric_clinical_cols = ['age', 'POD', 'RA diameter', 'shunt diameter']

    for col in binary_cols:
        if col in df_processed.columns:
            df_processed[col] = df_processed[col].astype(str).str.lower().map(mapping_dict)

    for col in numeric_clinical_cols:
        if col in df_processed.columns:
            df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')

    categorical_cols = ['HD(a, b, c)']
    valid_categorical = [c for c in categorical_cols if c in df_processed.columns]
    if valid_categorical:
        df_processed = pd.get_dummies(df_processed, columns=valid_categorical, drop_first=True, dtype=float)

    acoustic_cols = [col for col in df_processed.columns if re.match(r"^E\d+", col)]
    for col in acoustic_cols:
        df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce').fillna(0)

    y = df_processed[TARGET_VARIABLE]
    groups = df_processed['Pt No']

    train_indices, test_indices = split_data_by_patient(
        df_processed, TARGET_VARIABLE, test_size=0.2, random_state=UNIFIED_RANDOM_STATE
    )

    # Preserve pre-imputation copy for use in nested CV (fold-level imputation inside)
    df_train_for_cv = df_processed.loc[train_indices].copy()

    for col in binary_cols:
        if col in df_processed.columns and df_processed[col].isnull().any():
            df_processed[col].fillna(df_processed.loc[train_indices, col].mode()[0], inplace=True)
    for col in numeric_clinical_cols:
        if col in df_processed.columns and df_processed[col].isnull().any():
            df_processed[col].fillna(df_processed.loc[train_indices, col].mean(), inplace=True)

    print_patient_ids_summary(df_processed, train_indices, test_indices, TARGET_VARIABLE, OUTPUT_DIR_P2)

    # --- Feature selection ---
    df_train = df_processed.loc[train_indices]
    y_train = y.loc[train_indices]
    groups_train = groups.loc[train_indices]

    e_features = [col for col in df_processed.columns if re.match(r"^E\d+", col)]
    base_clinical = ['age', 'POD', 'men', 'arterial calcification', 'DM', 'HTN',
                    'Heart disease', 'RA diameter', 'shunt diameter', 'tabaco site', 'left']
    ohe_cols = [col for col in df_processed.columns if any(cat in col for cat in valid_categorical)] if valid_categorical else []
    clinical_cols = [c for c in base_clinical + ohe_cols if c in df_processed.columns]
    combined_pool = e_features + clinical_cols

    rf_selector = RandomForestClassifier(random_state=UNIFIED_RANDOM_STATE, class_weight='balanced')

    print("\nSelecting features for [E_only]:")
    e_only_cols = select_stable_features_cached(
        df_train, y_train, groups_train, e_features,
        rf_selector, N_FEATURES_TO_SELECT, N_BOOTSTRAPS_STABILITY, UNIFIED_RANDOM_STATE,
        OUTPUT_DIR_P2, f"{TARGET_COLUMN_FOR_LABELING}_{GROUP_THRESHOLD}",
        "Phase2", "E_only"
    )

    print("\nSelecting features for [Acoustic_Plus_Clinical]:")
    combined_cols = select_stable_features_cached(
        df_train, y_train, groups_train, combined_pool,
        rf_selector, N_FEATURES_TO_SELECT, N_BOOTSTRAPS_STABILITY, UNIFIED_RANDOM_STATE,
        OUTPUT_DIR_P2, f"{TARGET_COLUMN_FOR_LABELING}_{GROUP_THRESHOLD}",
        "Phase2", "Acoustic_Plus_Clinical"
    )

    feature_sets_p2 = {"E_only": e_only_cols, "Acoustic_Plus_Clinical": combined_cols}
    print(f"\nE_only: {e_only_cols}")
    print(f"Combined: {combined_cols}")

    # === Nested CV ===
    nested_cv_results_p2 = {}
    if RUN_MODE in ['full', 'cv_only']:
        print("\n--- Nested CV with HP Tuning ---")

        print("\n  Pool: E_only")
        pool_e = nested_cv_both_models(
            df_train_for_cv[e_features], y_train, groups_train,
            e_features, N_FEATURES_TO_SELECT, N_BOOTSTRAPS_STABILITY, UNIFIED_RANDOM_STATE
        )
        for mt in MODELS_TO_RUN:
            nested_cv_results_p2[f"{mt}_E_only"] = pool_e[mt]

        print("\n  Pool: Acoustic_Plus_Clinical")
        pool_c = nested_cv_both_models(
            df_train_for_cv[combined_pool], y_train, groups_train,
            combined_pool, N_FEATURES_TO_SELECT, N_BOOTSTRAPS_STABILITY, UNIFIED_RANDOM_STATE
        )
        for mt in MODELS_TO_RUN:
            nested_cv_results_p2[f"{mt}_Acoustic_Plus_Clinical"] = pool_c[mt]

        print("\n--- Nested CV Results ---")
        for label, res in nested_cv_results_p2.items():
            perf = res['perf']
            print(f"{label}: CV-AUC={perf.loc['auc','mean']:.3f}±{perf.loc['auc','std']:.3f}, "
                  f"Acc={perf.loc['accuracy','mean']:.3f}±{perf.loc['accuracy','std']:.3f}")

        print("\n--- Per-Fold Details ---")
        for label, res in nested_cv_results_p2.items():
            print(f"\n  {label}:")
            for d in res['details']:
                print(f"    Fold {d['fold']}: AUC={d['auc']:.3f}, HP=[{d['hp']}], Features={d['features']}")

    if RUN_MODE == 'cv_only':
        print("\n### PHASE 2 CV-ONLY COMPLETE ###")
        return

    # === Test evaluation ===
    if RUN_MODE in ['full', 'test_only']:
        print("\n--- Test Evaluation with HP-tuned Models ---")
        final_results_p2 = {}
        summary_data_p2 = []
        y_pred_probas_test_p2 = {}
        hp_summary_p2 = []

        y_test_global = y.loc[test_indices]
        test_patient_ids_global = groups.loc[test_indices]

        for feature_name, feature_list in feature_sets_p2.items():
            X_selected = df_processed[feature_list]
            X_train = X_selected.loc[train_indices]
            X_test = X_selected.loc[test_indices]
            y_train_loop = y.loc[train_indices]
            y_test_loop = y.loc[test_indices]
            groups_test = groups.loc[test_indices]

            for model_type in MODELS_TO_RUN:
                model_id = f"{model_type}_{feature_name}"
                print(f"\nProcessing: {model_id}")

                if model_type == 'RF':
                    final_model, best_params, oob = tune_rf_with_oob(
                        X_train, y_train_loop, RF_PARAM_GRID, UNIFIED_RANDOM_STATE
                    )
                    hp_info = f"n_est={best_params['n_estimators']}, depth={best_params['max_depth']}, leaf={best_params['min_samples_leaf']}, OOB-AUC={oob:.3f}"
                else:
                    final_model, best_C = tune_lr_with_gridsearch(
                        X_train, y_train_loop, groups_train, LR_PARAM_GRID, UNIFIED_RANDOM_STATE
                    )
                    hp_info = f"C={best_C}"

                print(f"  Best HP: {hp_info}")
                hp_summary_p2.append({'Model': model_id, 'Best HP': hp_info})

                proba_test = final_model.predict_proba(X_test)[:, 1]
                y_pred_probas_test_p2[model_id] = pd.Series(proba_test, index=X_test.index)

                ci_lower, ci_upper = calculate_bootstrap_ci(y_test_loop, proba_test, groups_test)

                row = {
                    'Model': model_id,
                    'Features': len(feature_list),
                    'AUC (95% CI)': f"{ci_lower:.3f} - {ci_upper:.3f}",
                }
                if model_id in nested_cv_results_p2:
                    cv_perf = nested_cv_results_p2[model_id]['perf']
                    row['CV-AUC'] = f"{cv_perf.loc['auc', 'mean']:.3f} ± {cv_perf.loc['auc', 'std']:.3f}"
                    row['CV-Accuracy'] = f"{cv_perf.loc['accuracy', 'mean']:.3f} ± {cv_perf.loc['accuracy', 'std']:.3f}"
                    row['CV-Sensitivity'] = f"{cv_perf.loc['sensitivity', 'mean']:.3f} ± {cv_perf.loc['sensitivity', 'std']:.3f}"
                    row['CV-Specificity'] = f"{cv_perf.loc['specificity', 'mean']:.3f} ± {cv_perf.loc['specificity', 'std']:.3f}"
                    row['CV-F1'] = f"{cv_perf.loc['f1_score', 'mean']:.3f} ± {cv_perf.loc['f1_score', 'std']:.3f}"
                summary_data_p2.append(row)

                final_results_p2[model_id] = {
                    'model': final_model, 'X_test': X_test,
                    'y_test': y_test_loop, 'X_train': X_train
                }

        table_p2_df = pd.DataFrame(summary_data_p2)
        print("\n" + "="*50)
        print("--- Phase 2: Model Performance Summary ---")
        print("="*50)
        print(table_p2_df.to_string(index=False))

        print("\n--- Best Hyperparameters ---")
        print(pd.DataFrame(hp_summary_p2).to_string(index=False))

        display_test_auc_summary(
            final_results_p2, y_pred_probas_test_p2,
            y_test_global, test_patient_ids_global, "Phase 2"
        )

        # AUC Difference
        y_pred_probas_df_p2 = pd.DataFrame(y_pred_probas_test_p2)
        table_diff_p2 = {}
        for prefix in MODELS_TO_RUN:
            pair = (f'{prefix}_Acoustic_Plus_Clinical', f'{prefix}_E_only')
            if all(p in y_pred_probas_df_p2.columns for p in pair):
                res = compute_auc_diff_paired(y_test_global, y_pred_probas_df_p2, test_patient_ids_global, pair)
                p_delong = delong_roc_test(
                    y_test_global.values,
                    y_pred_probas_df_p2[pair[0]].values,
                    y_pred_probas_df_p2[pair[1]].values
                )
                res['p_value (DeLong)'] = p_delong
                table_diff_p2[prefix] = pd.DataFrame([res])

        if table_diff_p2:
            print("\n" + "="*70)
            print("--- Phase 2: AUC Difference ---")
            print("="*70)
            for prefix, df in table_diff_p2.items():
                print(f"\n--- {prefix} ---")
                print(df.to_string(index=False))

        # SHAP + Calibration
        shap_model = "RF_Acoustic_Plus_Clinical" if "RF_Acoustic_Plus_Clinical" in final_results_p2 else list(final_results_p2.keys())[0]
        if shap_model in final_results_p2:
            best_res = final_results_p2[shap_model]
            print(f"\nDetailed analysis: '{shap_model}'")
            analyze_and_plot_shap(
                best_res['model'], best_res['X_train'], best_res['X_test'],
                OUTPUT_DIR_P2, shap_model, f"{TARGET_COLUMN_FOR_LABELING}_Phase2"
            )
            plot_calibration_curve(
                best_res['model'], best_res['X_test'], y_test_global,
                shap_model, OUTPUT_DIR_P2, f"{TARGET_COLUMN_FOR_LABELING}_Phase2"
            )

        plot_roc_curves(
            final_results_p2, table_p2_df,
            OUTPUT_DIR_P2 / f'ROC_Curve_{TARGET_COLUMN_FOR_LABELING}_Phase2.svg'
        )

        save_tables_to_word(
            None, None, table_p2_df, None, table_diff_p2,
            OUTPUT_DIR_P2 / f'Tables_{TARGET_COLUMN_FOR_LABELING}_Phase2.docx',
            TARGET_VARIABLE
        )

    print("\n" + "="*80)
    print("### PHASE 2 ANALYSIS COMPLETE ###")
    print("="*80)


In [ ]:
# =======================
# Main Execution
# =======================
if __name__ == "__main__":
    overall_start_time = time.time()

    main_phase1()
    main_phase2()

    overall_end_time = time.time()
    print(f"\nTotal analysis time: {(overall_end_time - overall_start_time) / 60:.2f} minutes")

In [ ]:
## =======================================================
# Robustness Validation of Model Performance Comparison
# ========================================================

import sys
import pandas as pd
import numpy as np
import re
import warnings
from pathlib import Path
import time
from collections import Counter

if "google.colab" in sys.modules:
    !pip install python-docx statsmodels shap -q
    !apt-get -qq install -y fonts-liberation
    from google.colab import drive
    drive.mount('/content/drive')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

warnings.filterwarnings('ignore')

# ==================================
# Configuration
# ==================================
TARGET_COLUMN_FOR_LABELING = 'FV'  # 'FV' or 'RI'
GROUP_THRESHOLD = 400              # FV: 400, RI: 0.6

DRIVE_BASE_PATH = Path("/content/drive/MyDrive/")  # "/path/to/your/data" for local
INPUT_EXCEL_PATH = DRIVE_BASE_PATH / "your_data.xlsx"

N_BOOTSTRAPS_ROBUSTNESS = 1000
UNIFIED_RANDOM_STATE = 243

PHASE_TO_RUN = 1  # 1: Phase 1 only, 2: Phase 2 only, 3: Both

# ===========================================================
# # Feature sets + HP-tuned model classes (from results)
# ===========================================================
if TARGET_COLUMN_FOR_LABELING == 'FV':
    STABLE_FEATURES_ENGINEERED = [
        'C27_PSD200', 'B5_Mean_Amp_0-250Hz', 'A1_RMS_Energy',
        'B6_Mean_Amp_250-500Hz', 'B13_Peak_Amp_0-250Hz'
    ]
    STABLE_FEATURES_PERCEPTUAL = [
        'E34_MFCC_2', 'E47_Spectral_Contrast', 'E33_MFCC_1',
        'E44_MFCC_12', 'E36_MFCC_4'
    ]
    # HP from  Phase 1 results
    MODEL_PERCEPTUAL = RandomForestClassifier(
        n_estimators=100, max_depth=3, min_samples_leaf=3,
        random_state=UNIFIED_RANDOM_STATE, class_weight='balanced'
    )
    MODEL_ENGINEERED = RandomForestClassifier(
        n_estimators=300, max_depth=3, min_samples_leaf=1,
        random_state=UNIFIED_RANDOM_STATE, class_weight='balanced'
    )

    # Phase 2 (FV: same features, same models)
    STABLE_FEATURES_ACOUSTIC_ONLY = STABLE_FEATURES_PERCEPTUAL
    STABLE_FEATURES_COMBINED = STABLE_FEATURES_PERCEPTUAL
    MODEL_ACOUSTIC_ONLY = MODEL_PERCEPTUAL
    MODEL_COMBINED = MODEL_PERCEPTUAL

elif TARGET_COLUMN_FOR_LABELING == 'RI':
    STABLE_FEATURES_ENGINEERED = [
        'A4_Active_Signal_Ratio', 'B5_Mean_Amp_0-250Hz', 'C27_PSD200',
        'A1_RMS_Energy', 'C25_TMP'
    ]
    STABLE_FEATURES_PERCEPTUAL = [
        'E47_Spectral_Contrast', 'E34_MFCC_2', 'E37_MFCC_5',
        'E43_MFCC_11', 'E42_MFCC_10'
    ]
    # HP from Phase 1 results (updated for RI)
    MODEL_PERCEPTUAL = RandomForestClassifier(
        n_estimators=300, max_depth=3, min_samples_leaf=3,
        random_state=UNIFIED_RANDOM_STATE, class_weight='balanced'
    )
    MODEL_ENGINEERED = RandomForestClassifier(
        n_estimators=100, max_depth=None, min_samples_leaf=3,
        random_state=UNIFIED_RANDOM_STATE, class_weight='balanced'
    )

    # Phase 2
    STABLE_FEATURES_ACOUSTIC_ONLY = [
        'E47_Spectral_Contrast', 'E34_MFCC_2', 'E37_MFCC_5',
        'E43_MFCC_11', 'E42_MFCC_10'
    ]
    STABLE_FEATURES_COMBINED = [
        'E47_Spectral_Contrast', 'age', 'E34_MFCC_2',
        'E43_MFCC_11', 'E42_MFCC_10'
    ]
    MODEL_ACOUSTIC_ONLY = RandomForestClassifier(
        n_estimators=300, max_depth=3, min_samples_leaf=3,
        random_state=UNIFIED_RANDOM_STATE, class_weight='balanced'
    )
    MODEL_COMBINED = RandomForestClassifier(
        n_estimators=100, max_depth=3, min_samples_leaf=3,
        random_state=UNIFIED_RANDOM_STATE, class_weight='balanced'
    )

# =================================
# Functions
# =================================

def load_and_preprocess_data(file_path, target_column, threshold):
    df = pd.read_excel(file_path)
    target_variable = f'{target_column}>={threshold}'
    df[target_variable] = (df[target_column] >= threshold).astype(int)
    print(f"Loaded: {file_path.name} (n={len(df)} samples)")
    return target_variable, df


def split_data_by_patient(df, target_variable, test_size=0.2, random_state=None):
    unique_patients = df['Pt No'].unique()
    patient_labels = df.drop_duplicates(subset=['Pt No']).set_index('Pt No')[target_variable]
    train_patients, test_patients = train_test_split(
        unique_patients, test_size=test_size, random_state=random_state,
        stratify=patient_labels.reindex(unique_patients)
    )
    train_indices = df[df['Pt No'].isin(train_patients)].index
    test_indices = df[df['Pt No'].isin(test_patients)].index
    print(f"Patient-level split: Train={len(train_patients)}, Test={len(test_patients)}")
    print(f"Sample-level split:  Train={len(train_indices)}, Test={len(test_indices)}")
    return train_indices, test_indices


def robustness_validation(
    X_train_df, y_train, groups_train,
    X_test_df, y_test,
    model_class_A, model_class_B,
    features_A, features_B,
    name_A, name_B,
    n_bootstraps, random_state
):
    """
    Robustness validation with separate HP-tuned models for each pool.
    """
    print(f"\n{'='*60}")
    print(f"Robustness Validation: {name_A} vs {name_B}")
    print(f"  Model A: {model_class_A}")
    print(f"  Model B: {model_class_B}")
    print(f"  Bootstrap iterations: {n_bootstraps}")
    print(f"{'='*60}")

    auc_diffs = []
    rng = np.random.RandomState(random_state)
    unique_patients = groups_train.unique()
    n_patients = len(unique_patients)
    start_time = time.time()

    for i in range(n_bootstraps):
        if (i + 1) % 200 == 0:
            elapsed = time.time() - start_time
            print(f"  Iteration {i+1}/{n_bootstraps} ({elapsed:.0f}s elapsed)")

        try:
            sampled = rng.choice(unique_patients, size=n_patients, replace=True)
            idx = groups_train[groups_train.isin(sampled)].index

            if len(idx) == 0 or len(np.unique(y_train.loc[idx])) < 2:
                continue

            model_A = clone(model_class_A).fit(X_train_df.loc[idx, features_A], y_train.loc[idx])
            model_B = clone(model_class_B).fit(X_train_df.loc[idx, features_B], y_train.loc[idx])

            auc_A = roc_auc_score(y_test, model_A.predict_proba(X_test_df[features_A])[:, 1])
            auc_B = roc_auc_score(y_test, model_B.predict_proba(X_test_df[features_B])[:, 1])

            auc_diffs.append(auc_A - auc_B)

        except Exception:
            continue

    valid = len(auc_diffs)
    print(f"Completed: {valid}/{n_bootstraps} valid iterations")

    if valid < 20:
        print("WARNING: Too few valid iterations.")
        return None

    diffs = np.array(auc_diffs)

    return {
        'comparison': f"{name_A} - {name_B}",
        'mean_diff': np.mean(diffs),
        'ci_lower': np.percentile(diffs, 2.5),
        'ci_upper': np.percentile(diffs, 97.5),
        'superiority_prob': np.mean(diffs > 0)
    }


# ==========================
# Main Execution
# ==========================

print(f"\n{'='*60}")
print(f"Robustness Validation (HP-tuned models)")
print(f"Task: {TARGET_COLUMN_FOR_LABELING} >= {GROUP_THRESHOLD}")
print(f"Phase: {PHASE_TO_RUN}")
print(f"{'='*60}")

overall_start = time.time()

TARGET_VARIABLE, df_orig = load_and_preprocess_data(
    INPUT_EXCEL_PATH, TARGET_COLUMN_FOR_LABELING, GROUP_THRESHOLD
)

df_processed = df_orig.copy()

binary_cols = ['men', 'arterial calcification', 'DM', 'HTN',
               'Heart disease', 'tabaco site', 'left']
mapping_dict = {'y': 1, 'n': 0, 'male': 1, 'female': 0,
                '1': 1, '0': 0, '1.0': 1, '0.0': 0}
numeric_clinical_cols = ['age', 'POD', 'RA diameter', 'shunt diameter']

for col in binary_cols:
    if col in df_processed.columns:
        df_processed[col] = df_processed[col].astype(str).str.lower().map(mapping_dict)

for col in numeric_clinical_cols:
    if col in df_processed.columns:
        df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')

categorical_cols = ['HD(a, b, c)']
valid_categorical = [c for c in categorical_cols if c in df_processed.columns]
if valid_categorical:
    df_processed = pd.get_dummies(df_processed, columns=valid_categorical,
                                   drop_first=True, dtype=float)

acoustic_cols = [col for col in df_processed.columns if re.match(r"^[A-E]\d+", col)]
for col in acoustic_cols:
    df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce').fillna(0)

y = df_processed[TARGET_VARIABLE]
groups = df_processed['Pt No']

train_indices, test_indices = split_data_by_patient(
    df_processed, TARGET_VARIABLE, test_size=0.2, random_state=UNIFIED_RANDOM_STATE
)

for col in binary_cols:
    if col in df_processed.columns and df_processed[col].isnull().any():
        df_processed[col].fillna(df_processed.loc[train_indices, col].mode()[0], inplace=True)

for col in numeric_clinical_cols:
    if col in df_processed.columns and df_processed[col].isnull().any():
        df_processed[col].fillna(df_processed.loc[train_indices, col].mean(), inplace=True)

X_train = df_processed.loc[train_indices]
y_train = y.loc[train_indices]
groups_train = groups.loc[train_indices]
X_test = df_processed.loc[test_indices]
y_test = y.loc[test_indices]

results = {}

if PHASE_TO_RUN in [1, 3]:
    results['Phase 1'] = robustness_validation(
        X_train, y_train, groups_train,
        X_test, y_test,
        model_class_A=MODEL_PERCEPTUAL,
        model_class_B=MODEL_ENGINEERED,
        features_A=STABLE_FEATURES_PERCEPTUAL,
        features_B=STABLE_FEATURES_ENGINEERED,
        name_A="Perceptual",
        name_B="Engineered",
        n_bootstraps=N_BOOTSTRAPS_ROBUSTNESS,
        random_state=UNIFIED_RANDOM_STATE
    )

if PHASE_TO_RUN in [2, 3]:
    results['Phase 2'] = robustness_validation(
        X_train, y_train, groups_train,
        X_test, y_test,
        model_class_A=MODEL_COMBINED,
        model_class_B=MODEL_ACOUSTIC_ONLY,
        features_A=STABLE_FEATURES_COMBINED,
        features_B=STABLE_FEATURES_ACOUSTIC_ONLY,
        name_A="Combined",
        name_B="Acoustic Only",
        n_bootstraps=N_BOOTSTRAPS_ROBUSTNESS,
        random_state=UNIFIED_RANDOM_STATE
    )

# ==========================
# Summary
# ==========================
print(f"\n{'='*60}")
print("RESULTS SUMMARY")
print(f"{'='*60}")
print(f"Task: {TARGET_COLUMN_FOR_LABELING} >= {GROUP_THRESHOLD}")

for phase, res in results.items():
    if res is None:
        print(f"\n{phase}: Analysis failed")
        continue
    print(f"\n{phase}: {res['comparison']}")
    print(f"  Mean AUC Difference:    {res['mean_diff']:.3f}")
    print(f"  95% CI:                 [{res['ci_lower']:.3f}, {res['ci_upper']:.3f}]")
    print(f"  Superiority Probability: {res['superiority_prob']:.1%}")

    if res['ci_lower'] > 0:
        print(f"  Conclusion: Statistically robust advantage for {res['comparison'].split(' - ')[0]}")
    elif res['ci_upper'] < 0:
        print(f"  Conclusion: Statistically robust advantage for {res['comparison'].split(' - ')[1]}")
    else:
        print(f"  Conclusion: No statistically robust difference")

total_time = (time.time() - overall_start) / 60
print(f"\nTotal analysis time: {total_time:.1f} minutes")